In [1]:
import numpy as np
import pandas as pd
from scipy.stats import f_oneway, kruskal, alexandergovern, chi2_contingency

In [2]:
column_map = {'Q2' : 'A_easy','Q3' : 'A_enjoyable','Q4' : 'A_status','Q5' : 'A_action',
              'Q7' : 'B_easy','Q8' : 'B_enjoyable','Q9' : 'B_status','Q10' : 'B_action',
              'Q12' : 'C_easy','Q13' : 'C_enjoyable','Q14' : 'C_status','Q15' : 'C_action',
              'Q16' : 'most_preferred','Q17' : 'least_preferred',
              'Q18' : 'MP_why','Q19' : 'LP_why'}
results = pd.read_csv("../data/surveys/survey_results.csv")
results.dropna(axis=1, how='all', inplace=True)
results.set_index('response', inplace=True)
results.rename(columns=column_map, inplace=True)
# Drop bogus responses
results.drop(index=results.index[[3, 4, 8, 22]], inplace=True)
results.tail()

,A_easy,A_enjoyable,A_status,A_action,B_easy,B_enjoyable,B_status,B_action,C_easy,C_enjoyable,C_status,C_action,most_preferred,least_preferred,MP_why,LP_why
response,,,,,,,,,,,,,,,,
18,4,3,3,2,2,2,2,2,3,4,3,3,Prototype A - Traditional,Prototype B - SMS-Based,It's simple and focused on being exactly what ...,I like that I wouldn't have to download an app...
19,4,4,4,4,4,4,4,4,4,4,4,4,Prototype A - Traditional,Prototype C - Gamified,I prefer Prototype A (Traditional App) because...,I chose Prototype C (Gamified) as least-prefer...
20,4,3,4,4,2,2,4,1,1,2,1,1,Prototype A - Traditional,Prototype C - Gamified,It has the most clarity and consistency with w...,The gamified interface felt like its features ...
21,5,5,5,5,3,3,3,3,4,4,4,3,Prototype A - Traditional,Prototype B - SMS-Based,"Although the gamified interface seems fun, the...",SMS messaging can feel clunky and is harder to...
22,5,4,4,5,5,5,5,5,5,5,5,5,Prototype B - SMS-Based,Prototype A - Traditional,"first, love ALL of the solutions. I would take...","traditional isn't bad, it's just not as easy a..."


In [3]:
results = pd.concat([results, results.describe()])

In [4]:
results.to_csv("../data/surveys/normalized_results.csv")

In [5]:
results.tail(10)

,A_easy,A_enjoyable,A_status,A_action,B_easy,B_enjoyable,B_status,B_action,C_easy,C_enjoyable,C_status,C_action,most_preferred,least_preferred,MP_why,LP_why
21,5.000000,5.000000,5.000000,5.000000,3.000000,3.000000,3.000000,3.000000,4.000000,4.000000,4.000000,3.000000,Prototype A - Traditional,Prototype B - SMS-Based,"Although the gamified interface seems fun, the...",SMS messaging can feel clunky and is harder to...
22,5.000000,4.000000,4.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,Prototype B - SMS-Based,Prototype A - Traditional,"first, love ALL of the solutions. I would take...","traditional isn't bad, it's just not as easy a..."
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,NaN,NaN,NaN,NaN
mean,4.368421,4.052632,4.157895,3.947368,3.947368,3.684211,4.000000,3.789474,3.894737,4.157895,3.789474,3.684211,NaN,NaN,NaN,NaN
std,0.495595,0.705036,0.602140,1.078769,0.970320,1.157230,0.942809,1.228321,1.100239,0.834210,1.031662,1.157230,NaN,NaN,NaN,NaN
min,4.000000,3.000000,3.000000,2.000000,2.000000,2.000000,2.000000,1.000000,1.000000,2.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
25%,4.000000,4.000000,4.000000,3.500000,3.500000,3.000000,4.000000,3.000000,3.500000,4.000000,3.500000,3.000000,NaN,NaN,NaN,NaN
50%,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,NaN,NaN,NaN,NaN
75%,5.000000,4.500000,4.500000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,4.000000,4.500000,NaN,NaN,NaN,NaN
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,NaN,NaN,NaN,NaN


In [6]:
most_preferred = results['most_preferred'].value_counts()
least_preferred = results['least_preferred'].value_counts()

print(f"Most-Preferred: {most_preferred}\n\n")
print(f"Least-Preferred: {least_preferred}")


Most-Preferred: most_preferred
Prototype A - Traditional    9
Prototype B - SMS-Based      6
Prototype C - Gamified       4
Name: count, dtype: int64


Least-Preferred: least_preferred
Prototype C - Gamified       8
Prototype B - SMS-Based      7
Prototype A - Traditional    4
Name: count, dtype: int64


In [7]:
# Add votes from interviews
all_mp = most_preferred.values + [2, 2, 1]
all_mp

array([11,  8,  5])

In [8]:
# Reverse order to match MP, add votes from interviews
all_lp = np.flip(least_preferred.values) + [0, 2, 3]
all_lp

array([ 4,  9, 11])

In [9]:
observed = np.array([all_lp, all_mp])
observed

array([[ 4,  9, 11],
       [11,  8,  5]])

In [10]:
# CHI-SQUARE RESULTS
stat, p_value, dof, expected = chi2_contingency(observed)
stat, p_value, dof, expected

(np.float64(5.575490196078431),
 np.float64(0.06155986899875066),
 2,
 array([[7.5, 8.5, 8. ],
        [7.5, 8.5, 8. ]]))

In [ ]:

index = ['easy', 'enjoyable', 'status', 'action']
proto_a = pd.DataFrame(results.loc['mean', 'A_easy':'A_action'])
proto_b = pd.DataFrame(results.loc['mean', 'B_easy':'B_action'])
proto_c = pd.DataFrame(results.loc['mean', 'C_easy':'C_action'])
a_means = proto_a.values.flatten().astype(float)
b_means = proto_b.values.flatten().astype(float)
c_means = proto_c.values.flatten().astype(float)


array([4.36842105, 4.05263158, 4.15789474, 3.94736842])

In [12]:
# ANOVA RESULTS
anova = f_oneway(a_means, b_means, c_means)
krusk = kruskal(a_means, b_means, c_means)
alex = alexandergovern(a_means, b_means, c_means)

anova, krusk, alex

(F_onewayResult(statistic=np.float64(2.948780487804875), pvalue=np.float64(0.10353162246616451)),
 KruskalResult(statistic=np.float64(4.222517730496455), pvalue=np.float64(0.12108544019580576)),
 AlexanderGovernResult(statistic=np.float64(3.6638886515568414), pvalue=np.float64(0.16010197455766395)))